# Supervised Learning Course Assignment 2

| Name | ID |
|------|----|
|  Mohamed Beshr Al Sofi | 20230717  |


In [ ]:
# ── Setup & Data Loading ──────────────────────────────────────────────────
# Import required libraries and load the CIFAR-10 dataset.
# The dataset is split into 40 000 training, 10 000 validation,
# and 10 000 test images (32×32 RGB).

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import time

# Global plot style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.dpi':      120,
    'axes.titlesize':  14,
    'axes.labelsize':  12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)
# CIFAR-10 Class Names
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
'dog','frog','horse','ship','truck']
# Load Data
(x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()
# Fixed Split: 40,000 train / 10,000 val / 10,000 test
x_train = x_train_full[:40000].astype('float32')
y_train = y_train_full[:40000]
x_val = x_train_full[40000:].astype('float32')
y_val = y_train_full[40000:]
x_test = x_test.astype('float32')
print(f"Train: {x_train.shape} | Val: {x_val.shape} | Test: {x_test.shape}")
print(f"Pixel range: [{x_train.min()}, {x_train.max()}]")
print(f"Image shape: {x_train[0].shape}")

In [ ]:
# ── Helper Functions ──────────────────────────────────────────────────────
# train_and_evaluate() compiles, trains, and evaluates a Keras model, then
# reports test accuracy, test loss, and total training time.
# plot_curves() draws training / validation metric curves for comparison.

def train_and_evaluate(model, x_tr, y_tr, x_v, y_v, x_te, y_te, epochs=20, batch_size=128, extra_callbacks=None):
    cb = extra_callbacks if extra_callbacks else []
    start = time.time()

    # Determine loss type to handle labels correctly
    is_sparse = model.loss == 'sparse_categorical_crossentropy'
    y_tr_proc = y_tr if is_sparse else to_categorical(y_tr, 10)
    y_v_proc = y_v if is_sparse else to_categorical(y_v, 10)
    y_te_proc = y_te if is_sparse else to_categorical(y_te, 10)

    history = model.fit(x_tr, y_tr_proc,
                        validation_data=(x_v, y_v_proc),
                        epochs=epochs, batch_size=batch_size,
                        callbacks=cb, verbose=0
                        )

    elapsed = time.time() - start
    test_loss, test_acc = model.evaluate(x_te, y_te_proc, verbose=0)
    print(f"Test Acc: {test_acc:.4f} | Test Loss: {test_loss:.4f} | Time:{elapsed:.1f}s")
    return history, test_acc, test_loss, elapsed

def plot_curves(histories, labels, metric='val_accuracy', title='', ylabel=''):
    plt.figure(figsize=(10, 6))
    for h, lbl in zip(histories, labels):
        plt.plot(h.history[metric], label=lbl)
    plt.xlabel('Epoch')
    plt.ylabel(ylabel if ylabel else metric)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

## **Task2 -> build model**

### part A

In [ ]:
# ── Part A – Width Experiment ─────────────────────────────────────────────
# Build and train three CNN variants with increasing filter widths
# (small: 8/16, medium: 16/32, large: 24/48 filters per block).
# Each model has two Conv blocks followed by a Flatten + Dense head.

from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPool2D,Flatten,Dense

partA_models = []
for i in range(1,4):
    model = Sequential([
        Conv2D(filters= i * 8, kernel_size=3, padding = 'same', activation= 'relu', input_shape=(32, 32, 3)),
        Conv2D(filters= i * 8, kernel_size=3, padding = 'same', activation= 'relu'),

        MaxPool2D(pool_size=(2,2)),

        Conv2D(filters= i * 8 * 2, kernel_size=3, padding = 'same', activation= 'relu'),
        Conv2D(filters= i * 8 * 2, kernel_size=3, padding = 'same', activation= 'relu'),

        Flatten(),
        Dense(units=256, activation='relu'),
        Dense(10, activation = 'softmax')
    ])

    model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
    )
    print(model.summary())
    model.fit(x_train,y_train,epochs=20,batch_size=128,validation_data=(x_val,y_val))
    partA_models.append(model)


In [ ]:
# ── Part A – Evaluate Accuracy ───────────────────────────────────────────
# Run inference on the test set for each Part-A model and record accuracy.

from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

accuracy_scores = []
name_models = ['small','medium','large']
for i,model in enumerate(partA_models):
    y_pred = model.predict(x_test)
    y_pred = np.argmax(y_pred, axis=1)
    accuracy = accuracy_score(y_test, y_pred)
    accuracy_scores.append(accuracy)


In [ ]:
# ── Part A – Visualise Results ───────────────────────────────────────────
# Plot a bar chart comparing the test accuracy of the three CNN widths.

# Create the bar plot with different colors
plt.figure(figsize=(8, 5))
plt.bar(name_models, accuracy_scores, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
plt.title('Accuracy of Models on Test Data')
plt.xlabel('Model')
plt.ylabel('Accuracy Score')
plt.ylim(0, 1) # Accuracy scores are between 0 and 1
plt.show()

### Part C

In [ ]:
# ── Part C – Depth & Pooling Experiment ──────────────────────────────────
# Define three CNN architectures that explore increasing depth and the
# effect of Global Average Pooling (GAP) as a replacement for Flatten.
#   Model 1: 2 Conv blocks + Flatten (baseline)
#   Model 2: 3 Conv blocks + 1× GAP
#   Model 3: 4 Conv blocks + 2× GAP

from tensorflow.keras.layers import GlobalAveragePooling2D
partC_models = [
    Sequential([
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu', input_shape=(32, 32, 3)),
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),

        MaxPool2D(pool_size=(2,2)),

        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),

        Flatten(),
        Dense(units=256, activation='relu'),
        Dense(10, activation = 'softmax')]),

    Sequential([
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu', input_shape=(32, 32, 3)),
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),

        MaxPool2D(pool_size=(2,2)),

        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),

        MaxPool2D(pool_size=(2,2)),

        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),
        GlobalAveragePooling2D(),

        Dense(units=256, activation='relu'),
        Dense(10, activation = 'softmax')]),

    Sequential([
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu', input_shape=(32, 32, 3)),
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),
        MaxPool2D(pool_size=(2,2)),

        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),
        MaxPool2D(pool_size=(2,2)),

        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),
        MaxPool2D(pool_size=(2,2)),

        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),
        Conv2D(filters= 32, kernel_size=3, padding = 'same', activation= 'relu'),
        GlobalAveragePooling2D(),

        Dense(units=256, activation='relu'),
        Dense(10, activation = 'softmax')])
    ]

In [ ]:
# ── Part C – Compile & Train ──────────────────────────────────────────────
# Compile all Part-C models with Adam + sparse categorical cross-entropy
# and train for 20 epochs; store each training history for later analysis.

history = []
for i,model in enumerate(partC_models):
    model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
    )
    print(model.summary())
    history.append(model.fit(x_train,y_train,epochs=20,batch_size=128,validation_data=(x_val,y_val)))

In [ ]:
# ── Part C – Training / Validation Loss Curves ───────────────────────────
# Plot side-by-side loss curves (train vs. val) for all three Part-C models
# to inspect overfitting behaviour as depth increases.

# Depth loss Comparison
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
model_names = ['Model 1 (Flatten)', 'Model 2 (1xGAP)', 'Model 3 (2xGAP)']

for i, h in enumerate(history):
    axs[i].plot(h.history['loss'], label='Train Loss')
    axs[i].plot(h.history['val_loss'], label='Val Loss')
    axs[i].set_title(f'{model_names[i]} Loss')
    axs[i].set_xlabel('Epoch')
    axs[i].set_ylabel('Loss')
    axs[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Part C – Evaluate Test Accuracy ──────────────────────────────────────
# Compute and print the test-set accuracy for each Part-C model.

from sklearn.metrics import accuracy_score

accuracy_scores = []
for i, model in enumerate(partC_models):
    y_pred = model.predict(x_test)
    y_pred = np.argmax(y_pred, axis=1)
    accuracy = accuracy_score(y_test.flatten(), y_pred)
    accuracy_scores.append(accuracy)
    print(f"Model {i+1} Accuracy: {accuracy:.4f}")

In [ ]:
# ── Part C – Visualise Accuracy Comparison ───────────────────────────────
# Bar chart comparing the test accuracy of the three depth/pooling variants.

# Create the bar plot with different colors
plt.figure(figsize=(8, 5))
plt.bar(name_models, accuracy_scores, color=plt.cm.plasma(np.linspace(0.2, 0.8, len(name_models))))
plt.title('Accuracy of Models on Test Data')
plt.xlabel('Model')
plt.ylabel('Accuracy Score')
plt.ylim(0, 1)
plt.show()

## **Task4 -> Optimizers**



### Part A

In [ ]:
# ── Part D – Optimizer Experiment (Setup) ────────────────────────────────
# Define a dictionary of optimizers to compare:
# SGD, SGD+Momentum, AdaGrad, RMSProp, and Adam — all at the same LR.

# Define optimizer factories to avoid reusing optimizer objects across different models
optimizer_factories = {
    'SGD':keras.optimizers.SGD(learning_rate=0.001),
    'Momentum':keras.optimizers.SGD(learning_rate=0.001, momentum=0.9),
    'AdaGrad':keras.optimizers.Adagrad(learning_rate=0.001),
    'RMSProp':keras.optimizers.RMSprop(learning_rate=0.001),
    'Adam':keras.optimizers.Adam(learning_rate=0.001),
}

In [ ]:
# ── Part D – Train with Different Optimizers ─────────────────────────────
# Re-build the medium CNN architecture and train one copy per optimizer.
# Results (accuracy, loss, time) are stored for comparison.

medium_model_base = Sequential([
    layers.Input(shape=(32, 32, 3)),
    Conv2D(filters=32, kernel_size=3, padding='same', activation='relu'),
    Conv2D(filters=32, kernel_size=3, padding='same', activation='relu'),
    MaxPool2D(pool_size=(2,2)),
    Conv2D(filters=64, kernel_size=3, padding='same', activation='relu'),
    Conv2D(filters=64, kernel_size=3, padding='same', activation='relu'),
    Flatten(),
    Dense(units=256, activation='relu'),
    Dense(10, activation='softmax')
])

results = {}

for opt_name, optimizer in optimizer_factories.items():
    print(f"\n--- Training with Optimizer: {opt_name} ---")
    model = medium_model_base

    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'],
                  )

    history, test_acc, test_loss, elapsed = train_and_evaluate(
        model, x_train, y_train, x_val, y_val, x_test, y_test,
        epochs=20, batch_size=128
    )
    results[opt_name] = history

# Plot all curves together for comparison
plot_curves(list(results.values()), list(results.keys()), title='Optimizer Comparison - Validation Accuracy')

## Part B

In [ ]:
# ── Part D – Learning-Rate Sweep (Adam) ──────────────────────────────────
# Train the medium CNN with Adam at three learning rates
# (0.0001, 0.001, 0.01) to study the effect on convergence.

learning_rates = [0.0001,0.001,0.01]
results = {}

for i in learning_rates:
  print(f"Learning Rate: {i} for Adam optimizer")
  model = medium_model_base
  model.compile(optimizer=keras.optimizers.Adam(learning_rate=i),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'],
                )

  history, test_acc, test_loss, elapsed = train_and_evaluate(
        model, x_train, y_train, x_val, y_val, x_test, y_test,
        epochs=30, batch_size=128
    )
  results[i] = history

plot_curves(list(results.values()), list(results.keys()), title='Learning Rate Comparison - Validation Accuracy')